In [104]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing - msa_m2023

In [105]:
# read in data
msa_df = pd.read_excel("../data/raw/MSA_M2023_dl.xlsx")

In [106]:
# filter detailed O_GROUP
msa_df = msa_df[msa_df["O_GROUP"] == "detailed"]
msa_df = msa_df.reset_index(drop=True)

In [107]:
# convert * and # to NaN
msa_df.replace(r'[\*#]', np.nan, regex=True, inplace=True)

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,97.28,NaN,NaN,98490,135280,202340,NaN,NaN,NaN,NaN
1,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,37.19,57.85,86.97,34440,49430,77360,120330,180900,NaN,NaN
2,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,...,45.31,72.66,97.35,60130,74980,94240,151130,202490,NaN,NaN
3,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-2022,Sales Managers,...,44.89,62.57,87.76,46870,60760,93370,130140,182530,NaN,NaN
4,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-2032,Public Relations Managers,...,35.61,46.71,62.93,55000,63680,74070,97160,130900,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140513,79600,"Worcester, MA-CT",4,MA,0,Cross-industry,cross-industry,1235,53-7062,"Laborers and Freight, Stock, and Material Move...",...,19.99,22.59,27.13,33730,36870,41570,47000,56420,NaN,NaN
140514,79600,"Worcester, MA-CT",4,MA,0,Cross-industry,cross-industry,1235,53-7063,Machine Feeders and Offbearers,...,21.87,23.35,26.13,39040,41310,45490,48570,54340,NaN,NaN
140515,79600,"Worcester, MA-CT",4,MA,0,Cross-industry,cross-industry,1235,53-7064,"Packers and Packagers, Hand",...,16.25,17.97,21.93,31200,31200,33800,37370,45610,NaN,NaN
140516,79600,"Worcester, MA-CT",4,MA,0,Cross-industry,cross-industry,1235,53-7065,Stockers and Order Fillers,...,17.57,20.59,22.48,31390,34900,36550,42830,46750,NaN,NaN


In [108]:
# drop columns
msa_df = msa_df.drop(columns=['AREA_TYPE','NAICS','NAICS_TITLE','I_GROUP', 'OWN_CODE', 'O_GROUP', 'EMP_PRSE', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT', 'MEAN_PRSE', 'ANNUAL','HOURLY'])

In [109]:
# lowercase columns
msa_df.columns = msa_df.columns.str.lower()

In [110]:
# rename area
msa_df = msa_df.rename(columns={"area": "cbsa_code"})

In [111]:
# reassign variable types
msa_df["tot_emp"] = msa_df["tot_emp"].astype("Int64")

cols = [
    "jobs_1000", "h_mean", "a_mean",
    "h_pct10", "h_pct25", "h_median", "h_pct75", "h_pct90",
    "a_pct10", "a_pct25", "a_median", "a_pct75", "a_pct90"
]
msa_df[cols] = (
    msa_df[cols]
    .replace(r"[,*#]", "", regex=True)   # remove commas + special chars
    .apply(pd.to_numeric, errors="coerce")  # convert to float
)

In [112]:
msa_df.head()

,cbsa_code,area_title,prim_state,occ_code,occ_title,tot_emp,jobs_1000,h_mean,a_mean,h_pct10,h_pct25,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90
0,10180,"Abilene, TX",TX,11-1011,Chief Executives,40,0.561,115.39,240020.0,47.35,65.04,97.28,NaN,NaN,98490.0,135280.0,202340.0,NaN,NaN
1,10180,"Abilene, TX",TX,11-1021,General and Operations Managers,2290,31.404,47.35,98480.0,16.56,23.77,37.19,57.85,86.97,34440.0,49430.0,77360.0,120330.0,180900.0
2,10180,"Abilene, TX",TX,11-2021,Marketing Managers,120,1.675,55.91,116290.0,28.91,36.05,45.31,72.66,97.35,60130.0,74980.0,94240.0,151130.0,202490.0
3,10180,"Abilene, TX",TX,11-2022,Sales Managers,250,3.433,51.83,107810.0,22.54,29.21,44.89,62.57,87.76,46870.0,60760.0,93370.0,130140.0,182530.0
4,10180,"Abilene, TX",TX,11-2032,Public Relations Managers,30,0.420,42.76,88930.0,26.44,30.62,35.61,46.71,62.93,55000.0,63680.0,74070.0,97160.0,130900.0


# Preprocessing - rpp

In [113]:
# read in data
rpp_df = pd.read_csv("../data/raw/rpp_2023.csv", skiprows=3)

In [114]:
# filter only line code 1 (RPP for all items)
rpp_df = rpp_df[(rpp_df["LineCode"] == 1) | (rpp_df["LineCode"] == 3)]

In [115]:
# rename 2023 column to RPP for clarity
rpp_df.rename(columns={'2023': 'RPP'}, inplace=True)

In [116]:
# pivot dataset so that line code 1 and 3 are on same rows
rpp_df = rpp_df.pivot( index=["GeoFIPS", "GeoName"], columns="LineCode", values="RPP" ).reset_index()
rpp_df.columns.name = None

In [117]:
# rename columns for line codes 1 and 3
rpp_df = rpp_df.rename(columns={
    1: "rpp_all",
    3: "rpp_housing"
})

# rename geofips
rpp_df = rpp_df.rename(columns={"GeoFIPS": "cbsa_code"})

In [118]:
# lowercase columns
rpp_df.columns = rpp_df.columns.str.lower()

In [119]:
rpp_df.head()

,cbsa_code,geoname,rpp_all,rpp_housing
0,00000,United States,100.000,100.574
1,00999,United States (Nonmetropolitan Portion) *,88.347,58.886
2,10180,"Abilene, TX (Metropolitan Statistical Area)",90.394,73.631
3,10420,"Akron, OH (Metropolitan Statistical Area)",92.924,77.456
4,10500,"Albany, GA (Metropolitan Statistical Area)",86.491,50.046


In [120]:
# reassign variable types
rpp_df["cbsa_code"] = rpp_df["cbsa_code"].astype(int)
rpp_df["rpp_all"] = rpp_df["rpp_all"].astype(float)
rpp_df["rpp_housing"] = rpp_df["rpp_housing"].astype(float)

In [121]:
rpp_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 389 entries, 0 to 388
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cbsa_code    389 non-null    int64  
 1   geoname      389 non-null    str    
 2   rpp_all      389 non-null    float64
 3   rpp_housing  389 non-null    float64
dtypes: float64(2), int64(1), str(1)
memory usage: 12.3 KB


# Creating the jobs table

In [122]:
# extract df from msa_df
jobs_df = msa_df[[
    "cbsa_code",
    "occ_code",
    "tot_emp",
    "jobs_1000",
    "h_mean",
    "a_mean",
    "h_pct10",
    "h_pct25",
    "h_median",
    "h_pct75",
    "h_pct90",
    "a_pct10",
    "a_pct25",
    "a_median",
    "a_pct75",
    "a_pct90"
]].copy()

In [123]:
# sort for consistency
jobs_df = jobs_df.sort_values(["cbsa_code", "occ_code"]).reset_index(drop=True)

In [124]:
jobs_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 140518 entries, 0 to 140517
Data columns (total 16 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   cbsa_code  140518 non-null  int64  
 1   occ_code   140518 non-null  str    
 2   tot_emp    138428 non-null  Int64  
 3   jobs_1000  138428 non-null  float64
 4   h_mean     131298 non-null  float64
 5   a_mean     138907 non-null  float64
 6   h_pct10    131298 non-null  float64
 7   h_pct25    131074 non-null  float64
 8   h_median   130510 non-null  float64
 9   h_pct75    129626 non-null  float64
 10  h_pct90    127944 non-null  float64
 11  a_pct10    138907 non-null  float64
 12  a_pct25    138683 non-null  float64
 13  a_median   138103 non-null  float64
 14  a_pct75    137185 non-null  float64
 15  a_pct90    135365 non-null  float64
dtypes: Int64(1), float64(13), int64(1), str(1)
memory usage: 17.3 MB


In [125]:
# export to csv
jobs_df.to_csv("../data/processed/jobs.csv", index=False)

# Creating the occupations table

In [126]:
# extract df from msa_df
occupations_df = msa_df[[
    "occ_code",
    "occ_title",
]].drop_duplicates()

In [127]:
occupations_df = occupations_df.sort_values("occ_code")

In [128]:
occupations_df.head()

,occ_code,occ_title
0,11-1011,Chief Executives
1,11-1021,General and Operations Managers
485,11-1031,Legislators
1442,11-2011,Advertising and Promotions Managers
2,11-2021,Marketing Managers


In [129]:
# export to csv
occupations_df.to_csv("../data/processed/occupations.csv", index=False)

# Creating the area table

In [ ]:
# extract needed data
areas_df = msa_df[["cbsa_code", "area_title", "prim_state"]].copy()

In [ ]:
# remove duplicate rows
areas_df = areas_df.drop_duplicates()

In [ ]:
# ensure 1 cbsa per area
areas_df = areas_df.drop_duplicates(subset=["cbsa_code"])

In [ ]:
# reset index
areas_df = areas_df.reset_index(drop=True)

In [ ]:
areas_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   cbsa_code   396 non-null    int64
 1   area_title  396 non-null    str  
 2   prim_state  396 non-null    str  
dtypes: int64(1), str(2)
memory usage: 9.4 KB


In [ ]:
# export to csv
areas_df.to_csv("../data/processed/areas.csv", index=False)

# Creating the rpp table

In [136]:
# drop unnecessary column
rpp_df = rpp_df[["cbsa_code", "rpp_all", "rpp_housing"]].copy()

In [137]:
# ensure one row per cbsa
rpp_df = rpp_df.drop_duplicates(subset=["cbsa_code"])


In [138]:
rpp_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 389 entries, 0 to 388
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cbsa_code    389 non-null    int64  
 1   rpp_all      389 non-null    float64
 2   rpp_housing  389 non-null    float64
dtypes: float64(2), int64(1)
memory usage: 9.2 KB


In [ ]:
# export to csv
rpp_df.to_csv("../data/processed/rpp.csv", index=False)